## Importing libraries

In [ ]:
import numpy as np
import pandas as pd

import json

import glob
import os
from pathlib import Path

import pickle

## Importing data

In [ ]:
# Dictionary containing all the json files imported (json_file_name : json_content)
with open('../raw_data/pickles/dc-4-eumaco.pickle', 'rb') as file:
	json_files_dict = pickle.load(file)

# List of all the json files names
json_filenames = list(json_files_dict)

Shows a sample of the field where the useful data is stored 

In [ ]:
json_files_dict[json_filenames[0]]['transmissions']['SpO2Transmission'][0][0]

## Analysis

Among three transmission fields...

In [ ]:
fields_to_check = json_files_dict[json_filenames[0]]['transmissions'].keys()
list(fields_to_check)

... choose between one of them (`SpO2Transmission`, `WeightTransmission` and `BloodPressureTransmission`)

In [ ]:
field_to_check = 'SpO2Transmission'

### Definition of functions that returns a sample of a field in a json/dictionary

Variant where iterable are NOT analyzed recursively

In [ ]:
def show_structure(data):
    '''Takes in input a dictionary and returns a version of it where:
    - keys are kept unchanged
    - values are substituted with the type name (iterable are NOT analyzed recursively)'''

    result = {}

    for key, val in data.items():
        if isinstance(val, dict):
            result[key] = show_structure(val)  # Recursive call
        else:
            result[key] = type(val).__name__
    return result

Variant where iterable are analyzed recursively

In [ ]:
from collections.abc import Mapping, Sequence

def show_structure(data):
    '''Takes in input a dictionary and returns a version of it where:
    - keys are kept unchanged
    - values are substituted with the type name (iterable are analyzed recursively)'''
    if isinstance(data, Mapping):
        result = {}
        for key, val in data.items():
            result[key] = show_structure(val)
        return result

    if isinstance(data, Sequence) and not isinstance(data, (str, bytes)):
        result = []
        for item in data:
            result.append(show_structure(item))
        return result

    return type(data).__name__


Shows the function behaviour on a sample field

In [ ]:
show_structure(json_files_dict[json_filenames[0]]['transmissions'][field_to_check][0][0])

### Definition of function that returns overview of the consistency of a field among json files

In [ ]:
def overview_of_field(json_files_dict, json_filenames, field_to_check):
    '''
    Takes in input:
    - a list of json filenames
    - a dictionary {json_filename: json_file_content}
    - the transmission subfield to check

    Returns an overview dictionary {'different': {json_file: [counter, ...]}, 'missing': [json_files]}
    that tells how many instances of a field to check are different on a given json file or how many json files misses that field
    '''
    # overview = {'different': {json_file: [counter, ...]}, 'missing': [json_files]}
    overview = {'different': {}, 'missing': []}

    first = show_structure(json_files_dict[json_filenames[0]]['transmissions'][field_to_check][0][0])
    for json_name in json_filenames:
        if field_to_check not in json_files_dict[json_name]['transmissions'].keys():
            overview['missing'].append(json_name)
        else:
            for t_counter, t_instance in enumerate(json_files_dict[json_name]['transmissions'][field_to_check][0]):
                formatted_tr_inst = show_structure(t_instance)
                if first != formatted_tr_inst:
                    overview['different'].setdefault(json_name, []).append(t_counter)
    
    return overview

### k

In [ ]:
# overview_of_field(json_files_dict, json_filenames, field_to_check)

fields_overview = {}
for field in fields_to_check:
    fields_overview[field] = overview_of_field(json_files_dict, json_filenames, field)

fields_overview

### Check consistency among the useful fields of the transmissions

Write the "show structure" json to file

In [ ]:
# data = digger_json(
#     json_files_list[0]
#     )

# with open("campion.json", "w") as f:
#     json.dump(data, f, indent=4)